In [1]:
import sys
sys.path.insert(0, '../src')

In [2]:
from generator_multidim import MultidimSampleGenerator

gen = MultidimSampleGenerator()
sample = gen.generate_random_sample(sample_size=6, min_trace_length=5, max_trace_length=10, event_dimension=1, type_count=5)

for trace in sample._sample:
    print(trace)

aa; ad; ac; ad; ae;
ab; ad; ae; ab; ac; ab;
ac; ae; ab; ab; ae; aa; ad; ad; ac;
aa; ab; aa; ad; ab;
aa; ad; aa; aa; aa; ab; ac;
aa; aa; ad; ac; ab; ae; ae;


In [3]:
from testbench_helper_functions import match_algos

results, columns = match_algos(
    sample_list=sample._sample,
    results=[],
    iterations=1,
    j=1,
    mod="synt",
    file_path="../experiments/experiment_results/notebook_result.csv",
    discovery=["uni"]
)

for row in results:
    display(dict(zip(columns, row)))

/Users/Rosie/.pyenv/versions/3.10.12/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-10 14:34:40,273	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-06-10 14:34:43,128	INFO worker.py:2003 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 
/Users/Rosie/.pyenv/versions/3.10.12/lib/python3.10/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


['']
[{'dict_iter': {'$x0; $x0;': {0: {('ad',): 3},
                              1: {('ab',): 3},
                              2: {('ab',): 3,
                                  ('ac',): 8,
                                  ('ad',): 7,
                                  ('ae',): 4},
                              3: {('aa',): 2, ('ab',): 4},
                              4: {('aa',): 2},
                              5: {('aa',): 1, ('ae',): 6}},
                ';': {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0},
                'aa;': {3: 0, 4: 0, 5: 0},
                'aa; aa;': {3: 2, 4: 2, 5: 1},
                'ab;': {1: 0, 2: 2, 3: 1},
                'ab; ab;': {1: 3, 2: 3, 3: 4},
                'ac;': {2: 0},
                'ac; ac;': {2: 8},
                'ad;': {0: 1, 2: 6},
                'ad; ad;': {0: 3, 2: 7},
                'ae;': {2: 1, 5: 5},
                'ae; ae;': {2: 4, 5: 6}},
  'dictionary': {'$x0; $x0;': True},
  'last_print_time': 1781094883.649793,
  'matching

{'sample size': '6',
 'trace length': '7',
 'iteration': '1',
 'algorithm': 'uni',
 'time': '0.5637822151184082',
 'iterations': '1',
 'queryset size': '0',
 'queryset': set(),
 'mode': 'synt',
 'searchspace': 1,
 'type sum': '2',
 'pattern sum': '2',
 'max query length': '-1',
 'supported types': '1',
 'pattern types': '1.8333333333333333',
 'avg query length': '0'}

(pid=22178) Calling ray.init() again after it has already been called.


In [ ]:
# uni = DUC

In [4]:
from discovery_bu_multidim import _next_queries_multidim
from query_multidim import MultidimQuery

alphabet = list(sample.get_sample_supported_typeset())
query = MultidimQuery("ac ac ac")
query.set_pos_last_type_and_variable()

children = _next_queries_multidim(query, alphabet, max_query_length=5, patternset={0: set()})
print(alphabet)
for child in children:
    print(child._query_string)

['ae']
ac ac ac ae


In [5]:
patternset ={}
all_patternset = {}
domain_cnt = sample._sample_event_dimension
gen_event= ';' * domain_cnt
gen_event_list = [i for i in gen_event]
att_vsdb = sample.get_att_vertical_sequence_database()
sample_size = sample._sample_size
vsdb = {}

for domain, dom_vsdb in att_vsdb.items():
    patternset[domain] = set()
    all_patternset[domain] = {trace_id: set() for trace_id in range(sample_size)}
    for key, value in dom_vsdb.items():
        new_key = ''.join(gen_event_list[:domain] + [key] + gen_event_list[domain:])
        vsdb[new_key] = value
        if not False:
            for item in value.keys():
                if len(value[item]) >= 2:
                    all_patternset[domain][item].add(key)
                    patternset[domain].add(key)
# in the patternset we have all values that appear more than once in at least one trace
print(patternset)

{0: {'ac', 'ab', 'ad', 'ae', 'aa'}}


In [6]:
# import sys
# sys.path.insert(0, '../src')
# from discovery_bu_multidim import _next_queries_multidim
# from query_multidim import MultidimQuery
# from generator_multidim import MultidimSampleGenerator

alphabet = list(sample.get_sample_supported_typeset())
#patternset = {0: set()}
max_query_length = 4

root = MultidimQuery(";")
root.set_pos_last_type_and_variable()

current_layer = [root]
layer_idx = 0

while current_layer:
    print(f"--- Layer {layer_idx} ---")
    next_layer = []
    for query in current_layer:
        children = _next_queries_multidim(query, alphabet, max_query_length, patternset)
        child_strings = [c._query_string for c in children]
        print(f"  {query._query_string!r:20} -> {child_strings}")
        next_layer.extend(children)
    current_layer = next_layer
    layer_idx += 1


--- Layer 0 ---
  ';'                  -> ['$x0; $x0;', 'ae']
--- Layer 1 ---
  '$x0; $x0;'          -> ['$x0; $x1; $x1; $x0;', '$x0; $x1; $x0; $x1;', '$x0; $x0; $x1; $x1;', '$x0; $x0; $x0;', '$x0; ae $x0;', '$x0; $x0; ae']
  'ae'                 -> ['ae $x0; $x0;', 'ae ae']
--- Layer 2 ---
  '$x0; $x1; $x1; $x0;' -> []
  '$x0; $x1; $x0; $x1;' -> []
  '$x0; $x0; $x1; $x1;' -> []
  '$x0; $x0; $x0;'     -> ['$x0; $x0; $x0; $x0;', '$x0; ae $x0; $x0;', '$x0; $x0; ae $x0;', '$x0; $x0; $x0; ae']
  '$x0; ae $x0;'       -> ['$x0; ae ae $x0;', '$x0; ae $x0; ae']
  '$x0; $x0; ae'       -> ['$x0; $x0; ae ae']
  'ae $x0; $x0;'       -> ['ae $x0; $x0; $x0;', 'ae $x0; ae $x0;', 'ae $x0; $x0; ae']
  'ae ae'              -> ['ae ae $x0; $x0;', 'ae ae ae']
--- Layer 3 ---
  '$x0; $x0; $x0; $x0;' -> []
  '$x0; ae $x0; $x0;'  -> []
  '$x0; $x0; ae $x0;'  -> []
  '$x0; $x0; $x0; ae'  -> []
  '$x0; ae ae $x0;'    -> []
  '$x0; ae $x0; ae'    -> []
  '$x0; $x0; ae ae'    -> []
  'ae $x0; $x0; $x0;'  -> []
 

In [ ]:
from discovery_bu_multidim import _next_queries_multidim
from query_multidim import MultidimQuery

alphabet = list(sample.get_sample_supported_typeset())
query = MultidimQuery("A;")
query.set_pos_last_type_and_variable()

children = _next_queries_multidim(query, alphabet, max_query_length=5, patternset={})

for child in children:
    print(child._query_string)

KeyError: 0